# Modul 19: Sequenzmodelle, Autoencoder und Generierung | Lösungen

## Überblick

Sie erstellen zeitlich korrekte Sequenzfenster und vergleichen naive Baselines mit kleinen Conv1D-, SimpleRNN- und GRU-Modellen. Danach trainieren Sie einen dichten Denoising-Autoencoder, analysieren Rekonstruktionsfehler, erkennen ungewöhnliche Ziffern und untersuchen Interpolation im latenten Raum.

**Zugehörige Vorlesungen**

- **Sequenzmodelle**
- **Autoencoder und Generierung**

## Lernziele

Nach der Bearbeitung können Sie:

- Sequenzfenster mit passenden Batchformen, zeitlichen Splits und naiven Baselines vorbereiten.
- kleine Conv1D-, SimpleRNN- und GRU-Modelle CPU-freundlich trainieren und zeitlich auswerten.
- Autoencoder für Rekonstruktion und Denoising einsetzen sowie latente Darstellungen und Anomaliegrenzen kritisch prüfen.

## Geprüfte Fähigkeiten

- Zeitfenster, Persistenzbaseline, zeitliche Skalierung und Fehleranalyse
- Keras Conv1D, SimpleRNN, GRU, Masking und EarlyStopping
- MLP-Autoencoder, Rekonstruktionsfehler, Denoising, Anomalieschwelle und latente Interpolation

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle erzeugt eine lokale univariate Zeitreihe mit Trend, Saison und Rauschen und lädt außerdem die kleinen Digits-Bilder. Die Deep-Learning-Modelle sind bewusst klein und auf wenige Epochen begrenzt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, precision_score, recall_score, f1_score
from sklearn.neural_network import MLPRegressor

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)

# Lokale Zeitreihe mit langsamem Trend, zwei Perioden und Rauschen.
anzahl_zeitpunkte = 1800
zeit_index = np.arange(anzahl_zeitpunkte)
zeitreihe = (
    0.0015 * zeit_index
    + 1.2 * np.sin(2 * np.pi * zeit_index / 50)
    + 0.35 * np.sin(2 * np.pi * zeit_index / 13)
    + rng.normal(0.0, 0.18, size=anzahl_zeitpunkte)
).astype("float32")

train_ende = int(0.60 * anzahl_zeitpunkte)
val_ende = int(0.80 * anzahl_zeitpunkte)

# Die Skalierungsparameter stammen ausschließlich aus der frühen Trainingsperiode.
zeit_mittel = float(zeitreihe[:train_ende].mean())
zeit_std = float(zeitreihe[:train_ende].std())
zeitreihe_skaliert = (zeitreihe - zeit_mittel) / zeit_std

ziffern = load_digits()
digit_bilder = (ziffern.data.astype("float32") / 16.0)
digit_labels = ziffern.target.astype("int64")

print("Zeitreihe:", zeitreihe.shape)
print("Digits-Matrix:", digit_bilder.shape)

### Aufgabe 1: Zeitliche Fenster und Persistenzbaseline erstellen

Implementieren Sie `make_windows`, die für jeden Zielzeitpunkt die vorherigen 30 Werte als Eingabefenster und den nächsten Wert als Ziel erzeugt. Die Funktion erhält einen Zielbereich `[start, end)` und darf nur vergangene Werte verwenden.

Erstellen Sie Training, Validierung und Test anhand der vorgegebenen Zeitgrenzen. Ergänzen Sie die letzte Achse, sodass Keras die Form `(Batch, Schritte, Merkmale)` erhält. Berechnen Sie auf Validierung und Test eine Persistenzbaseline, die einfach den letzten Fensterwert vorhersagt. Berichten Sie MAE in skalierten und ursprünglichen Einheiten.

In [ ]:
def make_windows(series, start, end, window_size):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def make_windows(series, start, end, window_size):
    """Erzeugt Fenster, deren Zielindex im Bereich [start, end) liegt."""
    X_fenster = []
    y_ziele = []
    ziel_indizes = []
    erster_zielindex = max(start, window_size)
    for zielindex in range(erster_zielindex, end):
        fenster_start = zielindex - window_size
        X_fenster.append(series[fenster_start:zielindex])
        y_ziele.append(series[zielindex])
        ziel_indizes.append(zielindex)
    return (
        np.asarray(X_fenster, dtype="float32"),
        np.asarray(y_ziele, dtype="float32"),
        np.asarray(ziel_indizes, dtype=int),
    )

fensterlaenge = 30
X_seq_train, y_seq_train, index_train = make_windows(
    zeitreihe_skaliert, 0, train_ende, fensterlaenge
)
X_seq_val, y_seq_val, index_val = make_windows(
    zeitreihe_skaliert, train_ende, val_ende, fensterlaenge
)
X_seq_test, y_seq_test, index_test = make_windows(
    zeitreihe_skaliert, val_ende, anzahl_zeitpunkte, fensterlaenge
)

# Eine univariate Reihe besitzt pro Zeitschritt genau ein Merkmal.
X_seq_train = X_seq_train[..., np.newaxis]
X_seq_val = X_seq_val[..., np.newaxis]
X_seq_test = X_seq_test[..., np.newaxis]

print("Sequenzformen:", X_seq_train.shape, X_seq_val.shape, X_seq_test.shape)
assert X_seq_train.shape[1:] == (fensterlaenge, 1)

val_persistenz = X_seq_val[:, -1, 0]
test_persistenz = X_seq_test[:, -1, 0]

baseline_bericht = pd.DataFrame(
    {
        "Split": ["Validierung", "Test"],
        "MAE_skaliert": [
            mean_absolute_error(y_seq_val, val_persistenz),
            mean_absolute_error(y_seq_test, test_persistenz),
        ],
    }
)
baseline_bericht["MAE_Originaleinheit"] = baseline_bericht["MAE_skaliert"] * zeit_std
display(baseline_bericht.round(4))

> **Musterantwort und Interpretation**
>
> Mittelwert und Standardabweichung späterer Validierungs- und Testperioden wären Zukunftsinformationen. Besonders bei Trend oder Drift kann diese Information die Darstellung der Trainingsperiode an die Zukunft anpassen und die Bewertung optimistisch machen. Die Trainingsstatistik wird deshalb unverändert auf spätere Zeiträume angewendet.

### Aufgabe 2: Ein kleines Conv1D-Prognosemodell trainieren

Erstellen Sie ein Keras-Modell aus einer Conv1D-Schicht mit höchstens 16 Filtern und Kernelgröße 3, GlobalAveragePooling1D, einer kleinen Dense-Schicht und einer linearen Ausgabe. Trainieren Sie höchstens 20 Epochen mit Adam, MSE und MAE sowie EarlyStopping.

Vergleichen Sie Validierungs- und Test-MAE mit der Persistenzbaseline. Zeichnen Sie für die ersten 150 Testzeitpunkte Wahrheit, Baseline und Conv1D-Prognose in ursprünglichen Einheiten.

In [ ]:
# Eingabeform eines Beispiels: (fensterlaenge, 1)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(RANDOM_SEED)

conv1d_modell = keras.Sequential(
    [
        keras.Input(shape=(fensterlaenge, 1)),
        layers.Conv1D(16, kernel_size=3, padding="causal", activation="relu"),
        layers.GlobalAveragePooling1D(),
        layers.Dense(8, activation="relu"),
        layers.Dense(1),
    ],
    name="kleines_conv1d",
)
conv1d_modell.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"],
)
history_conv1d = conv1d_modell.fit(
    X_seq_train,
    y_seq_train,
    validation_data=(X_seq_val, y_seq_val),
    epochs=20,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
        )
    ],
    verbose=0,
)

conv_val = conv1d_modell.predict(X_seq_val, verbose=0).ravel()
conv_test = conv1d_modell.predict(X_seq_test, verbose=0).ravel()

conv_bericht = pd.DataFrame(
    {
        "Modell": ["Persistenz", "Conv1D"],
        "Val_MAE": [
            mean_absolute_error(y_seq_val, val_persistenz),
            mean_absolute_error(y_seq_val, conv_val),
        ],
        "Test_MAE": [
            mean_absolute_error(y_seq_test, test_persistenz),
            mean_absolute_error(y_seq_test, conv_test),
        ],
    }
)
conv_bericht[["Val_MAE_Original", "Test_MAE_Original"]] = (
    conv_bericht[["Val_MAE", "Test_MAE"]] * zeit_std
)
display(conv_bericht.round(4))

anzuzeigen = 150
plt.plot(index_test[:anzuzeigen], y_seq_test[:anzuzeigen] * zeit_std + zeit_mittel, label="Wahr")
plt.plot(index_test[:anzuzeigen], test_persistenz[:anzuzeigen] * zeit_std + zeit_mittel, label="Persistenz")
plt.plot(index_test[:anzuzeigen], conv_test[:anzuzeigen] * zeit_std + zeit_mittel, label="Conv1D")
plt.xlabel("Zeitindex")
plt.ylabel("Wert in Originaleinheiten")
plt.title("Zeitliche Testprognosen")
plt.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Conv1D teilt Filter über die Zeit und erkennt lokale wiederkehrende Muster effizient. Causal Padding verhindert, dass ein Filter innerhalb eines Fensters zukünftige Positionen relativ zu seinem Ausgabepunkt verwendet. GlobalAveragePooling verdichtet jedoch alle Positionen zu Mittelwerten und verliert genaue zeitliche Lageinformationen. Für Aufgaben, bei denen die jüngsten Schritte besonders wichtig sind, kann eine andere Verdichtung sinnvoller sein.

### Aufgabe 3: SimpleRNN und GRU unter gleichen Bedingungen vergleichen

Erstellen Sie zwei Modelle mit identischer Eingabe und ungefähr ähnlicher kleiner Größe:

- SimpleRNN mit 16 Einheiten,
- GRU mit 16 Einheiten.

Beide erhalten eine lineare Ausgabe und werden mit denselben Splits, Batchgrößen, maximalen Epochen und EarlyStopping-Regeln trainiert. Vergleichen Sie Parameterzahl, ausgeführte Epochen, Validierungs-MAE und Test-MAE. Wählen Sie das Modell ausschließlich anhand der Validierung und berichten Sie danach dessen Testfehler.

In [ ]:
def baue_rekurrentes_modell(art):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def baue_rekurrentes_modell(art):
    if art == "SimpleRNN":
        sequenz_schicht = layers.SimpleRNN(16)
    elif art == "GRU":
        sequenz_schicht = layers.GRU(16)
    else:
        raise ValueError("art muss 'SimpleRNN' oder 'GRU' sein.")

    modell = keras.Sequential(
        [
            keras.Input(shape=(fensterlaenge, 1)),
            sequenz_schicht,
            layers.Dense(1),
        ],
        name=art.lower(),
    )
    modell.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"],
    )
    return modell

rekurrente_berichte = []
rekurrente_modelle = {}
rekurrente_testprognosen = {}
for art in ["SimpleRNN", "GRU"]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_SEED)
    modell = baue_rekurrentes_modell(art)
    historie = modell.fit(
        X_seq_train,
        y_seq_train,
        validation_data=(X_seq_val, y_seq_val),
        epochs=20,
        batch_size=32,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=4,
                restore_best_weights=True,
            )
        ],
        verbose=0,
    )
    val_pred = modell.predict(X_seq_val, verbose=0).ravel()
    test_pred = modell.predict(X_seq_test, verbose=0).ravel()
    rekurrente_modelle[art] = modell
    rekurrente_testprognosen[art] = test_pred
    rekurrente_berichte.append(
        {
            "Modell": art,
            "Parameter": modell.count_params(),
            "Epochen": len(historie.history["loss"]),
            "Val_MAE": mean_absolute_error(y_seq_val, val_pred),
            "Test_MAE": mean_absolute_error(y_seq_test, test_pred),
        }
    )

rekurrente_tabelle = pd.DataFrame(rekurrente_berichte).sort_values("Val_MAE")
display(rekurrente_tabelle.round(4))

gewaehlte_art = rekurrente_tabelle.iloc[0]["Modell"]
print("Nach Validierung gewählt:", gewaehlte_art)
print(
    "Test-MAE in Originaleinheiten:",
    round(
        mean_absolute_error(y_seq_test, rekurrente_testprognosen[gewaehlte_art]) * zeit_std,
        4,
    ),
)

> **Musterantwort und Interpretation**
>
> Eine GRU berechnet mehrere Gates und einen Kandidatenzustand. Dafür werden mehrere Eingabe- und rekurrente Gewichtsmatrizen sowie Biasvektoren benötigt. Die Gates können relevante Informationen länger bewahren und unwichtige vergessen, erhöhen aber Rechenaufwand und Modellkomplexität.

### Aufgabe 4: Maskierung und zeitliche Fehleranalyse anwenden

Erzeugen Sie einen kleinen Klassifikationsdatensatz aus Sequenzen unterschiedlicher Länge zwischen 12 und 30. Klasse 0 soll überwiegend eine niedrige Frequenz, Klasse 1 eine höhere Frequenz enthalten. Füllen Sie alle Sequenzen rechts mit Nullen auf Länge 30 auf.

Trainieren Sie ein Modell aus `Masking(mask_value=0.0)`, GRU und Sigmoid. Teilen Sie die Sequenzen reproduzierbar und stratifiziert. Vergleichen Sie Test-Accuracy für kurze Sequenzen bis Länge 20 und längere Sequenzen. Erklären Sie, warum echte Messwerte nicht zufällig genau dem Maskierungswert entsprechen sollten.

In [ ]:
# Erzeugen Sie 400 variable Sequenzen und speichern Sie zusätzlich ihre ursprünglichen Längen.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

anzahl_sequenzen = 400
max_laenge = 30
variable_sequenzen = np.zeros((anzahl_sequenzen, max_laenge, 1), dtype="float32")
variable_labels = np.zeros(anzahl_sequenzen, dtype="float32")
variable_laengen = np.zeros(anzahl_sequenzen, dtype=int)

for i in range(anzahl_sequenzen):
    klasse = i % 2
    laenge = int(rng.integers(12, max_laenge + 1))
    lokale_zeit = np.arange(laenge, dtype="float32")
    frequenz = 0.08 if klasse == 0 else 0.22
    sequenz = np.sin(2 * np.pi * frequenz * lokale_zeit)
    sequenz += rng.normal(0.0, 0.08, size=laenge)
    # Kleiner Offset verhindert, dass echte Werte systematisch exakt null sind.
    sequenz += 0.001
    variable_sequenzen[i, :laenge, 0] = sequenz
    variable_labels[i] = klasse
    variable_laengen[i] = laenge

indices = np.arange(anzahl_sequenzen)
idx_train, idx_test = train_test_split(
    indices,
    test_size=0.25,
    stratify=variable_labels,
    random_state=RANDOM_SEED,
)

masken_modell = keras.Sequential(
    [
        keras.Input(shape=(max_laenge, 1)),
        layers.Masking(mask_value=0.0),
        layers.GRU(12),
        layers.Dense(1, activation="sigmoid"),
    ]
)
masken_modell.compile(
    optimizer=keras.optimizers.Adam(0.002),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
masken_modell.fit(
    variable_sequenzen[idx_train],
    variable_labels[idx_train],
    validation_split=0.20,
    epochs=15,
    batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=0,
)

masken_probs = masken_modell.predict(variable_sequenzen[idx_test], verbose=0).ravel()
masken_pred = (masken_probs >= 0.5).astype(int)
kurz = variable_laengen[idx_test] <= 20

laengen_bericht = pd.DataFrame(
    {
        "Teilgruppe": ["Alle", "Länge bis 20", "Länge über 20"],
        "Anzahl": [len(idx_test), int(kurz.sum()), int((~kurz).sum())],
        "Accuracy": [
            accuracy_score(variable_labels[idx_test], masken_pred),
            accuracy_score(variable_labels[idx_test][kurz], masken_pred[kurz]),
            accuracy_score(variable_labels[idx_test][~kurz], masken_pred[~kurz]),
        ],
    }
)
display(laengen_bericht.round(3))

> **Musterantwort und Interpretation**
>
> Masking kennzeichnet jeden Zeitschritt mit ausschließlich Maskierungswerten als ungültig. Ist null ein echter, bedeutungsvoller Messwert, kann das Modell reale Zeitpunkte fälschlich ignorieren. Dann sind eine separate Maskeninformation, ein unmöglicher Füllwert oder längenbasierte Sequenzverarbeitung geeigneter.

### Aufgabe 5: Einen dichten Denoising-Autoencoder trainieren

Teilen Sie die Digits-Daten reproduzierbar in Training, Validierung und Test. Entfernen Sie für das Autoencoder-Training alle Ziffern 9, damit sie später als unbekannte Muster dienen können.

Erzeugen Sie verrauschte Eingaben durch additives Gaußrauschen und Begrenzung auf 0 bis 1. Trainieren Sie einen `MLPRegressor` mit symmetrischer Architektur `(32, 12, 32)`, der aus verrauschten Bildern die sauberen Pixel rekonstruiert. Verwenden Sie Early Stopping und eine begrenzte Iterationszahl. Vergleichen Sie den Rekonstruktions-MSE auf sauberen und verrauschten Testeingaben und visualisieren Sie Original, verrauschte Eingabe und Rekonstruktion für fünf normale Ziffern.

In [ ]:
# Ziffer 9 wird nur aus dem Autoencoder-Training entfernt, nicht aus dem späteren Test.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

X_digit_train, X_digit_test, y_digit_train, y_digit_test = train_test_split(
    digit_bilder,
    digit_labels,
    test_size=0.25,
    stratify=digit_labels,
    random_state=RANDOM_SEED,
)
X_digit_train, X_digit_val, y_digit_train, y_digit_val = train_test_split(
    X_digit_train,
    y_digit_train,
    test_size=0.20,
    stratify=y_digit_train,
    random_state=RANDOM_SEED,
)

normal_train = y_digit_train != 9
normal_val = y_digit_val != 9
normal_test = y_digit_test != 9

X_ae_train = X_digit_train[normal_train]
X_ae_val = X_digit_val[normal_val]

rausch_std = 0.20
X_ae_train_noisy = np.clip(
    X_ae_train + rng.normal(0.0, rausch_std, X_ae_train.shape),
    0.0,
    1.0,
).astype("float32")
X_ae_val_noisy = np.clip(
    X_ae_val + rng.normal(0.0, rausch_std, X_ae_val.shape),
    0.0,
    1.0,
).astype("float32")

# MLPRegressor unterstützt Mehrfachausgaben. Jede der 64 Ausgaben rekonstruiert einen Pixel.
autoencoder = MLPRegressor(
    hidden_layer_sizes=(32, 12, 32),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=140,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=12,
    random_state=RANDOM_SEED,
    batch_size=64,
)
autoencoder.fit(X_ae_train_noisy, X_ae_train)
print("Trainingsiterationen:", autoencoder.n_iter_)

X_normal_test = X_digit_test[normal_test]
X_normal_test_noisy = np.clip(
    X_normal_test + rng.normal(0.0, rausch_std, X_normal_test.shape),
    0.0,
    1.0,
).astype("float32")
rekonstruktion_aus_sauber = np.clip(autoencoder.predict(X_normal_test), 0.0, 1.0)
rekonstruktion_aus_noisy = np.clip(autoencoder.predict(X_normal_test_noisy), 0.0, 1.0)

print("MSE aus sauberer Eingabe:", round(mean_squared_error(X_normal_test, rekonstruktion_aus_sauber), 5))
print("MSE aus verrauschter Eingabe:", round(mean_squared_error(X_normal_test, rekonstruktion_aus_noisy), 5))

for index in range(5):
    for titel, bild in [
        ("Original", X_normal_test[index]),
        ("Verrauscht", X_normal_test_noisy[index]),
        ("Rekonstruktion", rekonstruktion_aus_noisy[index]),
    ]:
        plt.figure()
        plt.imshow(bild.reshape(8, 8), cmap="gray", vmin=0, vmax=1)
        plt.title(titel)
        plt.axis("off")
        plt.show()

> **Musterantwort und Interpretation**
>
> Ein gewöhnlicher Autoencoder lernt vor allem, gegebene Eingaben über einen komprimierten Engpass zu rekonstruieren. Der latente Raum besitzt ohne zusätzliche Regularisierung keine garantierte einfache Wahrscheinlichkeitsverteilung. Beliebige latente Punkte können deshalb unrealistische Ausgaben erzeugen. Variational Autoencoder oder andere generative Modelle ergänzen stärkere Annahmen für das Sampling neuer Beispiele.

### Aufgabe 6: Integrationsaufgabe: Rekonstruktionsanomalien und latente Interpolation

Berechnen Sie für normale Validierungsbilder den mittleren quadratischen Rekonstruktionsfehler pro Bild und setzen Sie die Anomalieschwelle auf das 95. Perzentil. Wenden Sie diese Schwelle auf alle Testbilder an und behandeln Sie Ziffer 9 als positive Anomalieklasse.

Berichten Sie Precision, Recall und F1 der Anomalieerkennung und visualisieren Sie Beispiele mit besonders hohem Fehler. Berechnen Sie anschließend die 12-dimensionale Engpassdarstellung zweier normaler Testbilder manuell aus den gelernten Gewichtsmatrizen, interpolieren Sie fünf Zwischenpunkte und dekodieren Sie sie durch die verbleibenden Schichten. Kennzeichnen Sie klar, dass plausible Interpolation keine Garantie für gültige neue Daten ist.

In [ ]:
def relu_numpy(x):
    return np.maximum(0.0, x)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def rekonstruktionsfehler(originale, rekonstruktionen):
    return np.mean((originale - rekonstruktionen) ** 2, axis=1)

val_rekonstruktion = np.clip(autoencoder.predict(X_ae_val), 0.0, 1.0)
val_fehler = rekonstruktionsfehler(X_ae_val, val_rekonstruktion)
anomalie_schwelle = float(np.quantile(val_fehler, 0.95))

alle_test_rekonstruktionen = np.clip(autoencoder.predict(X_digit_test), 0.0, 1.0)
test_fehler = rekonstruktionsfehler(X_digit_test, alle_test_rekonstruktionen)
wahre_anomalie = (y_digit_test == 9).astype(int)
vorhergesagte_anomalie = (test_fehler > anomalie_schwelle).astype(int)

print("Anomalieschwelle:", round(anomalie_schwelle, 6))
print("Precision:", round(precision_score(wahre_anomalie, vorhergesagte_anomalie, zero_division=0), 3))
print("Recall:", round(recall_score(wahre_anomalie, vorhergesagte_anomalie, zero_division=0), 3))
print("F1:", round(f1_score(wahre_anomalie, vorhergesagte_anomalie, zero_division=0), 3))

hohe_fehler = np.argsort(test_fehler)[-6:][::-1]
for index in hohe_fehler:
    plt.figure()
    plt.imshow(X_digit_test[index].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    plt.title(f"Ziffer {y_digit_test[index]}, Rekonstruktionsfehler {test_fehler[index]:.4f}")
    plt.axis("off")
    plt.show()

# MLPRegressor mit drei Hidden-Schichten besitzt vier Gewichtsmatrizen.
W0, W1, W2, W3 = autoencoder.coefs_
b0, b1, b2, b3 = autoencoder.intercepts_


def kodiere_bis_engpass(X):
    h1 = relu_numpy(X @ W0 + b0)
    z = relu_numpy(h1 @ W1 + b1)
    return z


def dekodiere_ab_engpass(z):
    h3 = relu_numpy(z @ W2 + b2)
    ausgabe = h3 @ W3 + b3
    return np.clip(ausgabe, 0.0, 1.0)

normale_testindizes = np.flatnonzero(y_digit_test != 9)
index_a = normale_testindizes[0]
index_b = normale_testindizes[10]
z_a = kodiere_bis_engpass(X_digit_test[[index_a]])[0]
z_b = kodiere_bis_engpass(X_digit_test[[index_b]])[0]

for alpha in np.linspace(0.0, 1.0, 5):
    z_interp = (1.0 - alpha) * z_a + alpha * z_b
    bild_interp = dekodiere_ab_engpass(z_interp.reshape(1, -1))[0]
    plt.figure()
    plt.imshow(bild_interp.reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    plt.title(f"Latente Interpolation alpha={alpha:.2f}")
    plt.axis("off")
    plt.show()

> **Musterantwort und Interpretation**
>
> Ein Autoencoder kann auch unbekannte Muster überraschend gut rekonstruieren oder seltene, aber gültige normale Beispiele schlecht rekonstruieren. Die Schwelle hängt von der Validierungsverteilung und der gewünschten Fehlerfolge ab. Ziffer 9 ist hier nur ein kontrolliertes Lehrbeispiel. In einer realen Anwendung benötigen Schwellen eine fachliche Validierung, Teilgruppenprüfung, Überwachung bei Drift und einen Prozess für menschliche Nachprüfung.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?